<a href="https://colab.research.google.com/github/nattsukun/AI-Agent-Skills/blob/main/JaiTTS_F5TTS_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JaiTTS-F5TTS: Thai Voice Cloning (Colab)

Run inference with [JaiTTS-F5TTS](https://huggingface.co/JTS-AI/JaiTTS-F5TTS), a non-autoregressive Thai zero-shot voice cloning model based on F5-TTS.

- Paper: *JaiTTS: A Thai Voice Cloning Model* ([arXiv:2604.27607](https://arxiv.org/abs/2604.27607))
- Model: [huggingface.co/JTS-AI/JaiTTS-F5TTS](https://huggingface.co/JTS-AI/JaiTTS-F5TTS)
- Inference codebase adapted from [ThonburianTTS](https://github.com/biodatlab/thonburian-tts)

**Research prototype** — released for research and benchmarking only.

**Before you start:** In Colab, go to `Runtime > Change runtime type` and select a **GPU** (e.g. T4) for reasonable inference speed.

## 0. Check GPU

In [ ]:
!nvidia-smi

Sat Jul 25 19:10:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Clone the inference codebase
This model uses the `flowtts` pipeline adapted from ThonburianTTS.

In [ ]:
import os

if not os.path.isdir("thonburian-tts"):
    !git clone https://github.com/biodatlab/thonburian-tts.git

%cd thonburian-tts

import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

Cloning into 'thonburian-tts'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 214 (delta 66), reused 9 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (214/214), 2.77 MiB | 7.70 MiB/s, done.
Resolving deltas: 100% (66/66), done.
/content/thonburian-tts/thonburian-tts


## 2. Install dependencies
Installs from the repo's own `requirements.txt` (torch, f5-tts, pydub, vocos, pythainlp, etc.) so versions stay in sync with the `flowtts` pipeline code.

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 78.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.8/122.8 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q python-crfsuite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.0 MB/s eta 0:00:00


## 3. Upload a reference audio clip

Upload a short `.wav` (or other audio) file containing the voice you want to clone, along with an accurate transcription of what is said in it.

In [ ]:
from google.colab import files

uploaded = files.upload()
reference_audio_path = next(iter(uploaded.keys()))
print(f"Using reference audio: {reference_audio_path}")

Saving Recording (7).m4a to Recording (7).m4a
Using reference audio: Recording (7).m4a


## 4. Set reference transcription and text to generate
Edit the strings below. Leave `reference_text` as an empty string `""` to let the pipeline auto-transcribe your reference audio instead.

In [ ]:
reference_text = "ข้อความที่คุณพูดในคลิปเสียงเพื่อให้ AI เรียนรู้"  # @param {type:"string"}
gen_text = "ขอความที่จะให้ AI Generate"  # @param {type:"string"}

### 4b. AI-Powered Text Chunking
เราจะใช้ Gemini API เพื่อวิเคราะห์บริบทของภาษาไทยและแบ่งข้อความให้เป็นธรรมชาติที่สุด โดยพยายามให้แต่ละช่วงมีความยาวประมาณ 60-80 ตัวอักษร (ประมาณ 10 วินาที)

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json

# Configure Gemini
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel('gemini-2.5-flash')
except Exception as e:
    print("Please set GOOGLE_API_KEY in Colab Secrets (🔑 symbol on the left)")

def get_ai_chunks(text):
    prompt = f"""
    Task: Split the following Thai text into natural-sounding phrases for Text-to-Speech.
    Guidelines:
    1. Each phrase should be roughly 50-80 characters long (to aim for ~10 seconds of speech).
    2. Split only at natural breathing points, logical pauses, or ends of sentences.
    3. Do not change or remove any words.
    4. Return the result as a JSON list of strings.

    Text: {text}
    """
    response = model.generate_content(prompt)
    # Extract JSON from response
    try:
        cleaned_response = response.text.strip().replace('```json', '').replace('```', '')
        return json.loads(cleaned_response)
    except:
        print("AI fallback to simple splitting due to format error.")
        return [text] # Fallback

# Process the chunks
print("AI is analyzing and chunking text...")
ai_sentences = get_ai_chunks(gen_text)

print(f"AI split text into {len(ai_sentences)} natural chunks:")
for i, s in enumerate(ai_sentences):
    print(f"{i+1}: {s}")

AI is analyzing and chunking text...
AI split text into 9 natural chunks:
1: มีคลิปตอนแพ็กสินค้า แต่จำไม่ได้ว่าคลิปไหนเป็นของออเดอร์นี้
2: ต้องไล่ดูคลิปทีละไฟล์ อัปโหลดขึ้นไดรฟ์ แล้วตั้งค่าสิทธิ์ก่อนส่งอีก
3: นี่คือเหตุผลที่เราสร้าง ProofMe
4: แค่สแกนป้ายพัสดุ แล้วเริ่มแพ็ก ProofMe จะบันทึกวิดีโอ
5: และผูกหลักฐานเข้ากับหมายเลขออเดอร์หรือหมายเลขพัสดุให้ทันที
6: เมื่อเกิดปัญหา เพียงค้นหาจากหมายเลขพัสดุ เปิดดู และแชร์หลักฐานได้ทันที
7: ไม่ต้องตั้งชื่อไฟล์ ไม่ต้องไล่หาคลิป และไม่ต้องต่อคอมพิวเตอร์
8: ProofMe กำลังเปิดรับร้านค้าออนไลน์กลุ่มแรก เพื่อทดลองใช้กับออเดอร์จริง ฟรี
9: ProofMe หลักฐานที่พร้อมใช้ โดยไม่เพิ่มภาระ


หลังจากรันด้านบนแล้ว ใน Cell ถัดไป (Step 6) ให้เปลี่ยนการใช้ `sentences` เป็น `ai_sentences` เพื่อใช้ผลลัพธ์จาก AI ครับ

## 5. Load the JaiTTS-F5TTS pipeline

In [ ]:
import torch
from flowtts.inference import FlowTTSPipeline, ModelConfig, AudioConfig

model_config = ModelConfig(
    language="th",
    model_type="F5",
    checkpoint="hf://JTS-AI/JaiTTS-F5TTS/model.pt",
    vocab_file="hf://JTS-AI/JaiTTS-F5TTS/vocab.txt",
    vocoder="vocos",
    device="cuda" if torch.cuda.is_available() else "cpu",
)

audio_config = AudioConfig(
    silence_threshold=-45,
    cfg_strength=2.0, # Reduced slightly for stability
    nfe_step=32,
    speed=1.0,
)

# Initialize pipeline with explicit silence removal to avoid tensor mismatch
pipeline = FlowTTSPipeline(model_config=model_config, audio_config=audio_config)
# Force internal model to remove silence from reference which often fixes the tensor mismatch
pipeline.model.remove_silence = True

Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /root/.cache/huggingface/hub/models--JTS-AI--JaiTTS-F5TTS/snapshots/651e39bf13b2475ca5f381234128ed97727f8b7a/vocab.txt
token :  custom
model :  /root/.cache/huggingface/hub/models--JTS-AI--JaiTTS-F5TTS/snapshots/651e39bf13b2475ca5f381234128ed97727f8b7a/model.pt 



## 6. Generate speech

In [ ]:
import re
from pathlib import Path
from pydub import AudioSegment

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Ensure the temp directory used by the pipeline exists
Path("temp").mkdir(parents=True, exist_ok=True)

# Provide reference_text explicitly (step 4) if you can
ref_text = reference_text.strip() or None

# Use ai_sentences from the previous AI chunking step
# If ai_sentences doesn't exist yet, it will fallback to gen_text as a single chunk
try:
    sentences_to_process = ai_sentences
    print(f"Using {len(sentences_to_process)} chunks provided by AI.")
except NameError:
    print("Warning: ai_sentences not found. Using raw text instead.")
    sentences_to_process = [gen_text]

chunk_paths = []
for i, sentence in enumerate(sentences_to_process):
    print(f"Generating chunk {i+1}/{len(sentences_to_process)}: {sentence[:30]}...")
    chunk_path = pipeline(
        text=sentence,
        ref_voice=reference_audio_path,
        ref_text=ref_text,
        output_file=str(output_dir / f"chunk_{i:03d}.wav"),
        speed=audio_config.speed,
        check_duration=True,
    )
    chunk_paths.append(chunk_path)

# Stitch the per-sentence clips back together with a short pause between them.
gap = AudioSegment.silent(duration=150)
combined = AudioSegment.silent(duration=0)
for i, path in enumerate(chunk_paths):
    combined += AudioSegment.from_wav(path)
    if i < len(chunk_paths) - 1:
        combined += gap

output_path = str(output_dir / "output.wav")
combined.export(output_path, format="wav")
print(f"\nSaved combined audio to {output_path}")

Using 9 chunks provided by AI.
Generating chunk 1/9: มีคลิปตอนแพ็กสินค้า แต่จำไม่ได...
Converting audio...
Using custom reference text...

ref_text   สวัสดีครับ นี่คือเสียงตัวอย่างของผม ผมกำลังอ่านข้อความนี้ด้วยน้ำเสียงที่เป็นธรรมชาติ ชัดเจน และผ่อนคลาย ไม่เร็วหรือช้าจนเกินไป. 
gen_text 0 มีคลิปตอนแพ็กสินค้า แต่จำไม่ได้ว่าคลิปไหนเป็นของออเดอร์นี้


Generating audio in 1 batches...


100%|██████████| 1/1 [00:04<00:00,  4.43s/it]


Time taken is 4.80 seconds
Generating chunk 2/9: ต้องไล่ดูคลิปทีละไฟล์ อัปโหลดข...
Converting audio...
Using custom reference text...

ref_text   สวัสดีครับ นี่คือเสียงตัวอย่างของผม ผมกำลังอ่านข้อความนี้ด้วยน้ำเสียงที่เป็นธรรมชาติ ชัดเจน และผ่อนคลาย ไม่เร็วหรือช้าจนเกินไป. 
gen_text 0 ต้องไล่ดูคลิปทีละไฟล์ อัปโหลดขึ้นไดรฟ์ แล้วตั้งค่าสิทธิ์ก่อนส่งอีก


Generating audio in 1 batches...


100%|██████████| 1/1 [00:04<00:00,  4.84s/it]


Time taken is 5.32 seconds
Generating chunk 3/9: นี่คือเหตุผลที่เราสร้าง ProofM...
Converting audio...
Using custom reference text...

ref_text   สวัสดีครับ นี่คือเสียงตัวอย่างของผม ผมกำลังอ่านข้อความนี้ด้วยน้ำเสียงที่เป็นธรรมชาติ ชัดเจน และผ่อนคลาย ไม่เร็วหรือช้าจนเกินไป. 
gen_text 0 นี่คือเหตุผลที่เราสร้าง ProofMe


Generating audio in 1 batches...


...

Time taken is 5.04 seconds
Generating chunk 8/9: ProofMe กำลังเปิดรับร้านค้าออน...
Converting audio...
Using custom reference text...

ref_text   สวัสดีครับ นี่คือเสียงตัวอย่างของผม ผมกำลังอ่านข้อความนี้ด้วยน้ำเสียงที่เป็นธรรมชาติ ชัดเจน และผ่อนคลาย ไม่เร็วหรือช้าจนเกินไป. 
gen_text 0 ProofMe กำลังเปิดรับร้านค้าออนไลน์กลุ่มแรก เพื่อทดลองใช้กับออเดอร์จริง ฟรี


Generating audio in 1 batches...


100%|██████████| 1/1 [00:04<00:00,  4.56s/it]


Time taken is 4.90 seconds
Generating chunk 9/9: ProofMe หลักฐานที่พร้อมใช้ โดย...
Converting audio...
Using custom reference text...

ref_text   สวัสดีครับ นี่คือเสียงตัวอย่างของผม ผมกำลังอ่านข้อความนี้ด้วยน้ำเสียงที่เป็นธรรมชาติ ชัดเจน และผ่อนคลาย ไม่เร็วหรือช้าจนเกินไป. 
gen_text 0 ProofMe หลักฐานที่พร้อมใช้ โดยไม่เพิ่มภาระ


Generating audio in 1 batches...


100%|██████████| 1/1 [00:03<00:00,  3.54s/it]

Time taken is 3.89 seconds

Saved combined audio to outputs/output.wav


## 7. Play and download the result

In [ ]:
from IPython.display import Audio, display

display(Audio(output_path))

In [ ]:
from google.colab import files

files.download(output_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### Citation
```
@misc{karnjanaekarin2026jaittsthaivoicecloning,
      title={JaiTTS: A Thai Voice Cloning Model},
      author={Jullajak Karnjanaekarin and Pontakorn Trakuekul and Narongkorn Panitsrisit and Sumana Sumanakul and Vichayuth Nitayasomboon and Nithid Guntasin and Thanavin Denkavin and Attapol T. Rutherford},
      year={2026},
      eprint={2604.27607},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2604.27607},
}
```
Codebase adapted from [ThonburianTTS](https://github.com/biodatlab/thonburian-tts).